# Fine-tune Qwen2.5-3B cho CoT clinical reasoning

Giai doan 2.5c. Doc JSONL sinh boi `scripts/generate_finetune_data.py`, fine-tune LoRA,
eval BEFORE/AFTER, merge va export de upload len HuggingFace Hub.

Chay theo thu tu cell 1 den 9. Cell 6 (eval BEFORE) PHAI chay truoc Cell 7 (train),
neu khong se khong co baseline de so sanh.

## Cell 1 -- Install

In [ ]:
%%capture
!pip install transformers peft datasets bitsandbytes accelerate trl

## Cell 2 -- Paths

In [ ]:
# === Colab: mount Drive va doc JSONL ===
from google.colab import drive
drive.mount('/content/drive')
JSONL_PATH = "/content/drive/MyDrive/med_qa/cot_training.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/med_qa/cot_reasoner_qwen"

# === Kaggle (comment/uncomment) ===
# JSONL_PATH = "/kaggle/input/med-qa-finetune/cot_training.jsonl"
# OUTPUT_DIR = "/kaggle/working/cot_reasoner_qwen"

## Cell 3 -- Load va inspect

In [ ]:
import json
from collections import Counter
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

raw = load_jsonl(JSONL_PATH)
print(f"Total samples: {len(raw)}")
label_counts = Counter(r["metadata"]["gt_label"] for r in raw)
print(f"Label distribution: {dict(label_counts)}")
# Kiem tra class imbalance truoc khi train -- neu 1 class chiem qua nhieu,
# can can nhac class weighting hoac sample lai truoc khi fine-tune.

dataset = Dataset.from_list(raw)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

## Cell 4 -- Load Qwen 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", torch_dtype=torch.float16,
)
model.config.use_cache = False

## Cell 5 -- LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1-2% trainable (~20-40M / 3B)

## Cell 6 -- Eval BEFORE (baseline, chay truoc khi train)

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)

before_results = []
for sample in eval_dataset.select(range(min(20, len(eval_dataset)))):
    msgs = sample["messages"][:-1]
    out = pipe(msgs)[0]["generated_text"][-1]["content"]
    true = sample["messages"][-1]["content"]
    try:
        pred_label = json.loads(out).get("cot_label", "unknown")
        true_label = json.loads(true).get("cot_label", "unknown")
    except Exception:
        pred_label = "parse_error"
        true_label = "unknown"
    before_results.append({"pred": pred_label, "true": true_label})

before_acc = sum(r["pred"] == r["true"] for r in before_results) / len(before_results)
print(f"[BEFORE fine-tune] Accuracy on eval subset: {before_acc:.2%}")

## Cell 7 -- Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,        # T4: 2 | P100/A100: 4
    gradient_accumulation_steps=8,         # effective batch = 16
    learning_rate=2e-4, fp16=True,
    save_strategy="epoch", evaluation_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10, warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none", dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, args=training_args,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    max_seq_length=2048,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"[DONE] Model saved to {OUTPUT_DIR}")

## Cell 8 -- Eval AFTER (so sanh voi Cell 6)

In [ ]:
after_results = []
for sample in eval_dataset.select(range(min(20, len(eval_dataset)))):
    msgs = sample["messages"][:-1]
    out = pipe(msgs)[0]["generated_text"][-1]["content"]
    true = sample["messages"][-1]["content"]
    try:
        pred_label = json.loads(out).get("cot_label", "unknown")
        true_label = json.loads(true).get("cot_label", "unknown")
    except Exception:
        pred_label, true_label = "parse_error", "unknown"
    after_results.append({"pred": pred_label, "true": true_label})

after_acc = sum(r["pred"] == r["true"] for r in after_results) / len(after_results)
print(f"[BEFORE] {before_acc:.2%}  |  [AFTER] {after_acc:.2%}  |  delta: {after_acc - before_acc:+.2%}")

## Cell 9 -- Merge LoRA + export (dung de upload len pod)

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained(OUTPUT_DIR + "_merged")
tokenizer.save_pretrained(OUTPUT_DIR + "_merged")
print(f"[DONE] Merged model -> {OUTPUT_DIR}_merged")
# Upload thu muc nay len HuggingFace Hub hoac cloud storage de pod mount khi start